# Cryptocurrency Agent with Azure AI Agent Service

This notebook demonstrates how to interact with a Cryptocurrency Agent via the Azure AI SDK. It is designed to guide you step-by-step through the process, including:

- Setting up environment variables and dependencies
- Enabling telemetry and tracing
- Creating agent threads and sending messages
- Polling run status and extracting the final answer

Follow along as each cell explains a part of the process.

## Step 1: Setup and Dependencies

In this step, we install the prerequisite libraries and load environment variables.

In [ ]:
%pip install azure-ai-projects azure-identity
%pip install opentelemetry-api azure-monitor-opentelemetry 
%pip install python-dotenv rich

In [ ]:
# Import dependencies and load environment variables
import os
import time
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from rich import print  # for rich formatted output

load_dotenv()

# Initialize the project client using connection string from environment
project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(),
    conn_str=os.environ["AZURE_AI_AGENT_PROJECT_CONNECTION_STRING"],
)

print("[green]Dependencies loaded and client initialized.[/green]")

## Step 2: Enable Tracing and Telemetry

Here, we enable Azure Monitor tracing and additional instrumentation. This ensures all operations of the agent are monitored.

In [ ]:
# Import tracing libraries
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor

# Enable Azure Monitor tracing
application_insights_connection_string = project_client.telemetry.get_connection_string()
if not application_insights_connection_string:
    print("[bold red]Application Insights was not enabled for this project.[/bold red]")
    print("[bold red]Enable it via the 'Tracing' tab in your AI Foundry project page.[/bold red]")
    exit()

configure_azure_monitor(connection_string=application_insights_connection_string)

# Enable additional instrumentations
project_client.telemetry.enable()

# Setup a tracer for the current scenario
import os
scenario = os.path.basename(__file__) if '__file__' in globals() else 'notebook_session'
tracer = trace.get_tracer(__name__)

print("[green]Tracing and telemetry are enabled.[/green]")

## Step 3: Agent Initialization and Creating a Thread

Next, we initialize the Cryptocurrency Agent and create a new thread to communicate with it.

In [ ]:
with tracer.start_as_current_span(scenario):
    # Initialize the agent using the assistant ID from environment variables
    agent = project_client.agents.get_agent(
        assistant_id=os.environ["AZURE_AI_CRYPTO_AGENT_ID"]
    )
    print(f"[green]Fetching Cryptocurrency Agent with ID: {agent.id}[/green]")

    # Create a new thread
    thread = project_client.agents.create_thread()
    print(f"[green]Created thread, thread ID: {thread.id}[/green]")

    # Create a message with the query content
    message = project_client.agents.create_message(
        thread_id=thread.id,
        role="user",
        content=(
            "Find out the 5 most traded cryptocurrencies today."
            "If that currency was traded on various exchanges,"
            "calculate the sum of all trading volumes. "
            "Also list the exchanges traded on, in a separate column."
        ),
    )
    print(f"[green]Created message, message ID: {message.id}[/green]")

## Step 4: Running the Agent and Extracting the Final Answer

Here, we trigger the agent run, poll for its status until completion, and finally extract and print the final assistant answer.

In [ ]:
    # Create a run for the agent
    run = project_client.agents.create_run(thread_id=thread.id, assistant_id=agent.id)

    # Poll the run as long as the run status is queued or in progress
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
        print(f"[yellow]Run status: {run.status}[/yellow]")

    # Get all messages and print only the final assistant answer
    messages = project_client.agents.list_messages(thread_id=thread.id)
    final_answer = next(
        (msg['content'][0]['text']['value'] for msg in messages['data']
         if msg['role'] == 'assistant' and msg.get('assistant_id')),
        "No answer received"
    )
    print(f"[blue]Final answer: {final_answer}[/blue]")

## Summary

This notebook showcased how to integrate with and monitor a Cryptocurrency Agent using the Azure AI SDK. We configured telemetry, created a communication thread, sent a query, and finally extracted the agent's final response. Execute the cells in sequence for a comprehensive demonstration.